## Name: Blessing Adeniji
##  Degree: MSc Artifical Intelligence Online
## Capstone Project: AI-Generated Text Detection - Deepfakes
Final Step: Error Analysis - Study the texts the detectors get wrong.

This notebook takes the trained models, run them over test sets, and keeps them per-sample predictions, 
then compare lingusitic features (sentence length, vocabulary diversity, punctuation) of wrong vs right predictions.

Analysis groups:
- Encoder vs decoder on the same hard task (MAGE-trained, tested on RAID) - do the two architectures fail on the SAME texts?
- The worst collapse (Abstracts-trained model on MAGE, 51% acc) - what does total generalisation failure look like linguistically?
- The best model's remaining errors (MAGE-trained on MAGE, ~5% error) - what stays hard even in the best case?




In [1]:
# Step 1: Get per-sample predictions.
# trainer.evaluate() only returns overall scores, so this uses
# trainer.predict() instead, which returns the prediction for EVERY text.
# Saved per row: the text, the true label, and each model's predicted label.
# A row where prediction != true label is a misclassification -
# these rows are the raw material for the whole analysis.

import os, pandas as pd, numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding


In [3]:
# Load the RAID test set
raid_test_df = pd.read_csv("data_splits/RAID_test.csv")

# The two MAGE-trained models to compare
models_to_analyse = {
    "encoder": "models/ettin68m_mage_final",
    "decoder": "models/decoder_mage_final",   
}

# Results table with text and true label
predictions_df = raid_test_df[["text", "label"]].copy()

for name, path in models_to_analyse.items():
    # Load this model and tokenizer from disk
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSequenceClassification.from_pretrained(path)
    
    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=512)
    test_ds = Dataset.from_pandas(raid_test_df).map(tokenize, batched=True)

    # Prediction-only trainer
    trainer = Trainer(
        model=model, 
        args=TrainingArguments(output_dir="tmp_predict", per_device_eval_batch_size=32, report_to="none"), 
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer), 
    )

    # Predict() returns raw scores for every sample; argmax picks the predicted clas
    output = trainer.predict(test_ds)
    predictions_df[f"{name}_pred"] = np.argmax(output.predictions, axis=1)
    print(f"{name}: done - accuracy {(predictions_df[f'{name}_pred'] == predictions_df['label']).mean():.4f}")

# check the correctness in each model
predictions_df["encoder_correct"] = predictions_df["encoder_pred"] == predictions_df["label"]
predictions_df["decoder_correct"] = predictions_df["decoder_pred"] == predictions_df["label"]

# Save the full per-sample table
os.makedirs("error_analysis", exist_ok=True)
predictions_df.to_csv("error_analysis/mage_models_on_raid_predictions.csv", index=False)

# The headline question: do they fail on the SAME texts?
both_enc_dec_wrong = (~predictions_df["encoder_correct"] & ~predictions_df["decoder_correct"]).sum()
only_encoder_wrong = (~predictions_df["encoder_correct"] & predictions_df["decoder_correct"]).sum()
only_decoder_wrong = (predictions_df["encoder_correct"] & ~predictions_df["decoder_correct"]).sum()
both_enc_dec_right = (predictions_df["encoder_correct"] & predictions_df["decoder_correct"]).sum()

# Print results
print(f"\nBoth right: {both_enc_dec_right}")
print(f"Both wrong: {both_enc_dec_wrong}")
print(f"Only encoder wrong: {only_encoder_wrong}")
print(f"Only decoder wrong: {only_decoder_wrong}")

Loading weights:   0%|          | 0/120 [00:00<?, ?it/s]

Map:   0%|          | 0/44021 [00:00<?, ? examples/s]

encoder: done - accuracy 0.7715


Loading weights:   0%|          | 0/157 [00:00<?, ?it/s]

Map:   0%|          | 0/44021 [00:00<?, ? examples/s]

decoder: done - accuracy 0.7804

Both right: 32408
Both wrong: 8113
Only encoder wrong: 1946
Only decoder wrong: 1554


In [6]:
# Step 2: Lingusitic features of errors and correct predictions
# For each text, compute: word count, average sentence length, type-token, ratio (vocabulary diversity), and punctuation density.
# Compare the feature averages across the four outcome groups from step 1

# re = regular expression for regex - mini lanaguage for pattern matching in text.
import re

# Load the per-sample predictions from step1
df = pd.read_csv("error_analysis/mage_models_on_raid_predictions.csv")

# Feature functions
def word_count(text):
    # No. of word in text
    return len(str(text).split())

def avg_sentence_length(text):
    # Split on sentence-ending punctuation: average words per sentence
    sentences = [s for s in re.split(r"[.!?]+", str(text)) if s.strip()]
    if not sentences: 
        return 0
    return np.mean([len(s.split()) for s in sentences])

def type_token_ratio(text):
    # Unique words / total words - higer = more vared vocabulary
    words = str(text).lower().split()
    if not words:
        return 0
    return len(set(words)) / len(words)

def punctuation_density(text):
    # Punctuation marks per 100 characters
    text = str(text)
    if not text:
        return 0
    punctuation = len(re.findall(r"[.,;:!?\"'()\-]", text))
    return 100 * punctuation / len(text)

# Compute all four features for every text (44k rows)
df["word_count"] = df["text"].apply(word_count)
df["avg_sentence_length"] = df["text"].apply(avg_sentence_length)
df["type_token_ratio"] = df["text"].apply(type_token_ratio)
df["punctuation_density"] = df["text"].apply(punctuation_density)

# Assign each row to its outcome group
def outcome_group(row):
    if row["encoder_correct"] and row["decoder_correct"]:
        return "both_enc_dec_right"
    if not row["encoder_correct"] and not row["decoder_correct"]:
        return "both_enc_dec_wrong"
    if not row["encoder_correct"]:
        return "only_encoder_wrong"
    return "only_decoder_wrong"

df["group"] = df.apply(outcome_group, axis=1)

# Compare feature averages per group
summary = df.groupby("group")[["word_count", "avg_sentence_length", "type_token_ratio", "punctuation_density"]].mean().round(2)
print(summary)

# Splits by true label - are errors mostly on human or AI texts?
print("\nError counts by true label )0=human, 1=AI):")
print(df[df["group"] == "both_enc_dec_wrong"]["label"].value_counts())

# Save the table for further analysis
df.to_csv("error_analysis/raid_predictions_with_features.csv", index=False)

                    word_count  avg_sentence_length  type_token_ratio  \
group                                                                   
both_enc_dec_right      265.24                26.12              0.62   
both_enc_dec_wrong      275.70                23.32              0.61   
only_decoder_wrong      242.98                21.85              0.65   
only_encoder_wrong      237.87                23.99              0.65   

                    punctuation_density  
group                                    
both_enc_dec_right                 2.53  
both_enc_dec_wrong                 2.41  
only_decoder_wrong                 2.76  
only_encoder_wrong                 2.83  

Error counts by true label )0=human, 1=AI):
label
0    6389
1    1724
Name: count, dtype: int64


In [7]:
# Step 3: This is to sharpen step2's findings
# False-flag rate: what fraction of HUMAN texts do the models wrongly call AI?
# Fairer feature comparison: compare error vs correct texts WITHIN the same true label 
# signifance test (Mann-Whitney U) on the key feature difference
from scipy.stats import mannwhitneyu

df = pd.read_csv("error_analysis/raid_predictions_with_features.csv")

# False flag rates
human_texts = df[df["label"] == 0]
ai_texts = df[df["label"] == 1]

# Print results
print("Total human texts:", len(human_texts), "| Total AI texts:", len(ai_texts))
print(f"Encoder false-flag rate (human called AI): {(human_texts['encoder_pred'] == 1).mean():.3f}")
print(f"Decoder false-flag rate (human called AI): {(human_texts['decoder_pred'] == 1).mean():.3f}")
print(f"Encoder missed-AI rate (AI called human): {(ai_texts['encoder_pred'] == 0).mean():.3f}")
print(f"Decoder missed-AI rate (AI called human): {(ai_texts['decoder_pred'] == 0).mean():.3f}")

# Feature comparison within Human texts only
human_right = df[(df["label"] == 0) & (df["group"] == "both_enc_dec_right")]
human_wrong = df[(df["label"] == 0) & (df["group"] == "both_enc_dec_wrong")]

features = ["word_count", "avg_sentence_length", "type_token_ratio", "punctuation_density"]
print("\nHuman texts: correctly classified vs falsely flagged (means)")
comparison = pd.DataFrame({
    "correct (human)": human_right[features].mean().round(2),
    "false-flagged (human)": human_wrong[features].mean().round(2),
})
print(comparison)

# Significance tests
print("\nMann-Whitney U tests (human correct vs human false-flagged):")
for f in features:
    stat, p = mannwhitneyu(human_right[f], human_wrong[f])
    print(f"  {f}: p = {p:.2e} {'(significant)' if p < 0.01 else '(not significant)'}")



Total human texts: 22011 | Total AI texts: 22010
Encoder false-flag rate (human called AI): 0.330
Decoder false-flag rate (human called AI): 0.327
Encoder missed-AI rate (AI called human): 0.127
Decoder missed-AI rate (AI called human): 0.112

Human texts: correctly classified vs falsely flagged (means)
                     correct (human)  false-flagged (human)
word_count                    301.63                 284.68
avg_sentence_length            24.45                  22.19
type_token_ratio                0.63                   0.60
punctuation_density             2.59                   2.31

Mann-Whitney U tests (human correct vs human false-flagged):
  word_count: p = 3.12e-33 (significant)
  avg_sentence_length: p = 1.95e-200 (significant)
  type_token_ratio: p = 8.06e-131 (significant)
  punctuation_density: p = 2.44e-74 (significant)


In [ ]:
# Step 4 :
# The Abstracts-trained encoder scored 51.6% on MAGE (F1 0.07) - it called
# nearly everything "human". Question: what does total generalisation
# failure look like? What little did it still detect, and which
# generators' text did it catch vs miss?

# Load the MAGE test set
mage_test_df = pd.read_csv("data_splits/MAGE_test.csv")

# Load the Abstracts-trained encoder
model_path = "models/ettin68m_abstracts_final"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)
mage_test_ds = Dataset.from_pandas(mage_test_df).map(tokenize, batched=True)

# Prediction-only trainer
trainer = Trainer(
    model=model,
    args=TrainingArguments(output_dir="models/tmp_predict", per_device_eval_batch_size=32, report_to="none"),
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)

# Per-sample predictions
output = trainer.predict(mage_test_ds)
collapse_df = mage_test_df[["text", "label", "src"]].copy()   # 'src' names each text's generator
collapse_df["pred"] = np.argmax(output.predictions, axis=1)
collapse_df["correct"] = collapse_df["pred"] == collapse_df["label"]

print("Accuracy:", collapse_df["correct"].mean().round(4))
print("\nPrediction counts (0=human, 1=AI):")
print(collapse_df["pred"].value_counts())

# AI texts: how many did it actually catch?
ai_texts = collapse_df[collapse_df["label"] == 1].copy()
print(f"\nAI texts caught: {(ai_texts['pred'] == 1).sum()} of {len(ai_texts)}")

# Catch rate by raw source (top 15 largest sources)
print("\nCatch rate by source (top 15 by sample count):")
catch_by_src = ai_texts.groupby("src")["pred"].agg(["mean", "count"]).sort_values("count", ascending=False).head(15)
catch_by_src.columns = ["catch_rate", "n_samples"]
print(catch_by_src.round(3))

# Group by generator family (rough extraction from the src string)
def generator_family(src):
    src = str(src).lower()
    for fam in ["gpt-3.5", "gpt_j", "gpt_neox", "gpt2", "gpt", "bloom", "opt", "llama", "t0", "65b", "flan", "glm", "dolly"]:
        if fam in src:
            return fam
    return "other"

ai_texts["family"] = ai_texts["src"].apply(generator_family)
print("\nCatch rate by generator family:")
print(ai_texts.groupby("family")["pred"].agg(["mean", "count"]).sort_values("mean", ascending=False).round(3))

# Save for further analysis
collapse_df.to_csv("error_analysis/abstracts_model_on_mage_predictions.csv", index=False)

Loading weights:   0%|          | 0/120 [00:00<?, ?it/s]

Map:   0%|          | 0/27996 [00:00<?, ? examples/s]

Accuracy: 0.5161

Prediction counts (0=human, 1=AI):
pred
0    27396
1      600
Name: count, dtype: int64

AI texts caught: 525 of 13998

Catch rate by source (top 15 by sample count):
                                            catch_rate  n_samples
src                                                              
eli5_machine_continuation_opt_iml_max_1.3b       0.000         63
wp_machine_continuation_t0_3b                    0.000         62
xsum_machine_continuation_opt_6.7b               0.000         61
xsum_machine_continuation_opt_iml_max_1.3b       0.000         61
sci_gen_machine_continuation_gpt-3.5-trubo       0.567         60
roct_machine_continuation_65B                    0.000         60
xsum_machine_topical_gpt-3.5-trubo               0.254         59
yelp_machine_continuation_13B                    0.000         59
roct_machine_continuation_t0_11b                 0.000         59
hswag_machine_continuation_t0_11b                0.034         59
eli5_machine_continuati

In [9]:
# Step 5: 
# The MAGE-trained encoder is the best generaliser, yet still gets ~5%
# wrong on its own test set. Question: what stays hard even in the best
# case? Profile the errors by label, generator source, and linguistic
# features.

# Load the MAGE test set
mage_test_df = pd.read_csv("data_splits/MAGE_test.csv")

# Load the MAGE-trained encoder (the champion)
model_path = "models/ettin68m_mage_final"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)
mage_test_ds = Dataset.from_pandas(mage_test_df).map(tokenize, batched=True)

trainer = Trainer(
    model=model,
    args=TrainingArguments(output_dir="models/tmp_predict", per_device_eval_batch_size=32, report_to="none"),
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
)

# Per-sample predictions
output = trainer.predict(mage_test_ds)
champ_df = mage_test_df[["text", "label", "src"]].copy()
champ_df["pred"] = np.argmax(output.predictions, axis=1)
champ_df["correct"] = champ_df["pred"] == champ_df["label"]

print("Accuracy:", champ_df["correct"].mean().round(4))

#  Error direction: false flags vs missed AI 
errors = champ_df[~champ_df["correct"]]
print(f"\nTotal errors: {len(errors)} of {len(champ_df)}")
print("Errors by true label (0=human falsely flagged, 1=AI missed):")
print(errors["label"].value_counts())

# Which generators still slip through? (missed AI by family) 
def generator_family(src):
    src = str(src).lower()
    for fam in ["gpt-3.5", "gpt_j", "gpt_neox", "gpt2", "gpt", "bloom", "opt", "llama", "t0", "65b", "flan", "glm", "dolly"]:
        if fam in src:
            return fam
    return "other"

ai_texts = champ_df[champ_df["label"] == 1].copy()
ai_texts["family"] = ai_texts["src"].apply(generator_family)
print("\nMiss rate by generator family (1 - catch rate):")
miss_by_family = ai_texts.groupby("family")["pred"].agg(["count"])
miss_by_family["miss_rate"] = ai_texts.groupby("family")["pred"].apply(lambda p: (p == 0).mean())
print(miss_by_family.sort_values("miss_rate", ascending=False).round(3))

# Linguistic features: errors vs correct, within each label
champ_df["word_count"] = champ_df["text"].apply(word_count)
champ_df["avg_sent_len"] = champ_df["text"].apply(avg_sentence_length)
champ_df["ttr"] = champ_df["text"].apply(type_token_ratio)
champ_df["punct_density"] = champ_df["text"].apply(punctuation_density)

features = ["word_count", "avg_sent_len", "ttr", "punct_density"]
print("\nHuman texts - correct vs falsely flagged (means):")
h = champ_df[champ_df["label"] == 0]
print(pd.DataFrame({"correct": h[h["correct"]][features].mean().round(2),
                    "false-flagged": h[~h["correct"]][features].mean().round(2)}))

print("\nAI texts - caught vs missed (means):")
a = champ_df[champ_df["label"] == 1]
print(pd.DataFrame({"caught": a[a["correct"]][features].mean().round(2),
                    "missed": a[~a["correct"]][features].mean().round(2)}))

champ_df.to_csv("error_analysis/mage_champion_on_mage_predictions.csv", index=False)

Loading weights:   0%|          | 0/120 [00:00<?, ?it/s]

Map:   0%|          | 0/27996 [00:00<?, ? examples/s]

Accuracy: 0.9503

Total errors: 1392 of 27996
Errors by true label (0=human falsely flagged, 1=AI missed):
label
0    838
1    554
Name: count, dtype: int64

Miss rate by generator family (1 - catch rate):
          count  miss_rate
family                    
65b         470      0.115
other      3461      0.061
gpt-3.5    1132      0.040
flan       2358      0.038
opt        4064      0.031
glm         437      0.027
bloom       431      0.012
t0          928      0.010
gpt_neox    354      0.006
gpt_j       363      0.003

Human texts - correct vs falsely flagged (means):
               correct  false-flagged
word_count      203.90           61.2
avg_sent_len     16.79           14.8
ttr               0.70            0.8
punct_density     2.97            3.0

AI texts - caught vs missed (means):
               caught  missed
word_count     223.23  104.70
avg_sent_len    18.81   16.44
ttr              0.71    0.78
punct_density    2.70    2.75
